# vLLM check

Runs 200 real requests from the trace through vLLM on a T4, at two KV-cache memory levels. Reports mean/p99 latency and preemption count. Use Runtime > Change runtime type > T4 GPU.

In [ ]:
!pip install -q uv
!uv venv /content/venv --python 3.11 -q
!uv pip install -q --python /content/venv/bin/python vllm==0.6.3.post1 pandas
!uv pip install -q --python /content/venv/bin/python setuptools
!uv pip install -q --python /content/venv/bin/python "transformers==4.45.2"

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/Azure/AzurePublicDataset/master/data/AzureLLMInferenceTrace_conv.csv

In [ ]:
%%writefile run_vllm.py
import json, sys
import numpy as np, pandas as pd
from vllm import LLM, SamplingParams

mem = float(sys.argv[1])
N = 200
d = pd.read_csv("AzureLLMInferenceTrace_conv.csv").iloc[:N]

# word count as a rough stand-in for token count
prompts = [" ".join(["word"] * int(c)) for c in d["ContextTokens"]]
max_tokens = d["GeneratedTokens"].astype(int).tolist()
max_len = int((d["ContextTokens"] + d["GeneratedTokens"]).max()) + 64

llm = LLM(model="Qwen/Qwen2.5-0.5B-Instruct", dtype="half",
          gpu_memory_utilization=mem, max_model_len=max_len, enforce_eager=True)
sampling = [SamplingParams(max_tokens=mt, ignore_eos=True) for mt in max_tokens]
outs = llm.generate(prompts, sampling)

lat = np.array([o.metrics.finished_time - o.metrics.arrival_time for o in outs])
pre = llm.llm_engine.scheduler[0].num_cumulative_preemption
res = {"gpu_mem": mem, "mean_s": round(float(lat.mean()), 3),
       "p99_s": round(float(np.percentile(lat, 99)), 3), "preemptions": int(pre)}
print("RESULT", json.dumps(res))
json.dump(res, open(f"result_{mem}.json", "w"))

In [ ]:
# normal memory level
!/content/venv/bin/python run_vllm.py 0.9 2>&1 | grep -E "RESULT|Error" | tail -5

In [ ]:
# low memory level, forces preemption
LOW = 0.25
!/content/venv/bin/python run_vllm.py {LOW} 2>&1 | grep -E "RESULT|Error" | tail -5

In [ ]:
import json, pandas as pd
pd.DataFrame([json.load(open(f"result_{m}.json")) for m in (0.9, LOW)])